# E20 因果入门：合成双重差分（DiD）

> 覆盖矩阵：因果推断入门（观测数据陷阱）

## 你将学到
1. 相关 ≠ 因果；平行趋势假设。
2. 用合成面板做 2×2 DiD。
3. **缺口**：无因果模块；本课仅教学，不宣称包已具备因果能力。


In [ ]:
QUICK = True
RANDOM_STATE = 42
from wanglaoshi.runtime import ready
ready(seed=RANDOM_STATE)


## 1. 合成面板：处理组后期受干预

In [ ]:
import numpy as np
import pandas as pd
from wanglaoshi.analyze import DataAnalyzer

rng = np.random.default_rng(RANDOM_STATE)
rows = []
true_effect = 5.0
for unit in range(80):
    treated = int(unit >= 40)
    for period in (0, 1):  # 0 pre, 1 post
        base = 20 + 3 * period + 2 * treated
        tau = true_effect if (treated == 1 and period == 1) else 0.0
        y = base + tau + rng.normal(0, 1.5)
        rows.append(dict(unit=unit, treated=treated, post=period, y=y))
df = pd.DataFrame(rows)
print(df.groupby(['treated', 'post'])['y'].mean().unstack())
print(DataAnalyzer(df).get_basic_info().get('shape'))


## 2. DiD 回归

In [ ]:
import statsmodels.formula.api as smf
from pathlib import Path

df['did'] = df['treated'] * df['post']
model = smf.ols('y ~ treated + post + did', data=df).fit()
print(model.summary().tables[1])
est = float(model.params['did'])
print('DiD estimate', round(est, 3), 'true', true_effect)
out = Path('outputs/20_causal_did'); out.mkdir(parents=True, exist_ok=True)
pd.DataFrame({'did_est': [est], 'true_effect': [true_effect]}).to_csv(out / 'did_summary.csv', index=False)


## 3. 错误对照：只看处理后截面相关

In [ ]:
post = df[df.post == 1]
print('post-only mean diff', post.loc[post.treated==1, 'y'].mean() - post.loc[post.treated==0, 'y'].mean())
print('（含组间固有差异，高估/混淆真效应）')


## 附录
- 作业：破坏平行趋势（给处理组不同 pre 斜率）再估 DiD，观察偏差。
- 包优化：长期可考虑文档化「因果不是基线工坊核心」；或独立 extras 教程，避免 API 承诺过度。
